树其实是一种特殊的图。但树有很强的限制，比如：
- 只有一个根
- 不能有环
- 父子关系比较明确
- 节点之间不是随便连的
而图更自由，它关注的是：**“谁和谁之间有关系？”**
比如：
- 城市 A 和城市 B 有路相连
- 课程 A 是课程 B 的先修课
- 用户 A 关注了用户 B
- 网页 A 链接到网页 B
所以，图本质上是在描述：对象之间的连接关系。
## 组成
### 顶点（Vertex）
顶点也叫 节点（Node），它是图中的基本单位。
每个顶点通常有：
- 一个标识名（key / id）
- 可能还有额外信息（payload）

> 可以把顶点简单理解为 **图中被研究的对象本身**。
### 边（Edge）
边也叫 弧（Arc），表示两个顶点之间存在某种关系。

### 有向图与无向图
> 在离散数学中都很清楚，这里就不再赘述

### 权重（Weight）
本质上是在描述 **从一个顶点到另一个顶点，需要付出的代价**。
### 路径（Path）与环（Cycle）
路径就是从一个顶点出发，沿着边一步一步走，形成的一串顶点序列。
路径长度有两种理解方式：
- 无权路径长度：只数边的条数。
  - `0 -> 5 -> 2 -> 3`共有 3 条边，所以长度是 3
- 带权路径长度：把路径上所有边的权重加起来。
  - `0 -> 5 权重 2 5 -> 2 权重 1 2 -> 3 权重 9`那么总长度就是12

环就是一条路径从某个顶点出发，最后又回到了这个顶点。

### 无环图（Acyclic Graph）
如果一个图里没有任何环，就叫无环图
如果是有向图且无环，就叫 **有向无环图（DAG, Directed Acyclic Graph）**
DAG 在算法里特别重要，因为很多“依赖关系问题”都可以建模成 DAG：
- 课程先修关系
- 任务调度
- 编译依赖
- 工作流执行顺序
因为如果有环，就会出现逻辑矛盾：
- A 依赖 B
- B 依赖 C
- C 又依赖 A

这就没法开始了。所以 DAG 很适合表示“必须按顺序完成”的问题。


## 抽象数据类型（Graph ADT）
| 序号 | 操作功能 | Python 语法 / 方法 | 详细说明 |
| ---- | -------- | ----------------- | -------- |
| 1 | 创建图 | `Graph()` | 初始化，创建一个空的图结构 |
| 2 | 添加顶点 | `addVertex(vert)` | 向图中插入单个顶点 |
| 3 | 添加无权重有向边 | `addEdge(fromVert, toVert)` | 添加一条从 `fromVert` 指向 `toVert` 的有向边 |
| 4 | 添加带权重有向边 | `addEdge(fromVert, toVert, weight)` | 添加一条带权重的、从 `fromVert` 指向 `toVert` 的有向边 |
| 5 | 获取单个顶点 | `getVertex(vertKey)` | 根据顶点名称/编号，查询并返回对应顶点 |
| 6 | 获取全部顶点 | `getVertices()` | 返回图中所有顶点的集合/列表 |
| 7 | 判断顶点是否存在 | `vert in graph` | 布尔判断：顶点存在返回 `True`，不存在返回 `False` |

### 存储方式
#### 邻接矩阵（Adjacency Matrix）
假设图中有 n 个顶点，就建立一个 n × n 的矩阵。矩阵中：
- 行表示起点
- 列表示终点
如果从 v 到 w 有边，就在 matrix[v][w] 里存值：
- 无权图
    - 有边：存 1
    - 没边：存 0
- 带权图
    - 有边：存权重
    - 没边：存 0 / 无穷 / 特殊值

| 分类 | 核心要点 | 详细说明 |
| :--- | :--- | :--- |
| **优点** | 实现简单 | 底层本质是二维数组，数据结构直观、代码易编写理解 |
|  | 判断两点连通极快 | 查询顶点v、w是否有边，直接读取 `matrix[v][w]`；<br>时间复杂度：$O(1)$ |
| **缺点** | 空间开销大、浪费严重 | 顶点数量为 $|V|$ 时，矩阵固定占用 $|V|^2$ 存储空间；<br>整体空间复杂度：$O(|V|^2)$ |
|  | 不适合稀疏图 | 现实业务大多是**稀疏图（边少点多）**；<br>矩阵里大量格子为空，存在巨额存储冗余，资源利用率极低 |

> 因此稠密图（Dense Graph）更适合用它存

#### 邻接表（Adjacency List）
这才是实际开发和算法里更常见的表示方式。
图里每个顶点，都维护一个“和自己直接相连的顶点列表”。也就是说：
- 图对象负责管理所有顶点
- 每个顶点自己记录“我能连到谁”

邻接表更适合稀疏图，因为它只存“真实存在的边”。空间复杂度大致是 $O(|V|+|E|)$ < $O(|V|^2)$ ，对稀疏图节省很多。

| 分类 | 核心要点 | 详细说明 |
| :--- | :--- | :--- |
| **优点** | 极度节省存储空间 | 专为**稀疏图**设计，只存储真实存在的边；<br>无冗余空数据，空间利用率极高 |
|  | 快速查询所有相邻顶点 | 获取某个顶点的全部邻居节点，直接读取该顶点的邻接列表即可，遍历高效便捷 |
| **缺点** | 判断两点连通效率偏低 | 想要确认顶点 `v` 与 `w` 是否直接相连；<br>需要遍历 `v` 的邻居列表逐个比对，无法做到 $O(1)$ 瞬时查询 |



In [ ]:
class Vertex:
    def __init__(self, key):
        self.id = key           # 表示我是谁，即顶点的编号/名字
        self.connectedTo = {}   # 字典，方便存储与我相连的顶点和权重

    def addNeighbor(self, nbr, weight=0):
        self.connectedTo[nbr] = weight  # 从当前顶点出发的边

    def __str__(self):
        return str(self.id) + ' connectedTo: ' + str([x.id for x in self.connectedTo])

    def getConnections(self):
        return self.connectedTo.keys()  # 返回与我相连的所有顶点

    def getId(self):
        return self.id

    def getWeight(self, nbr):
        return self.connectedTo[nbr]

In [ ]:
class Graph:
    def __init__(self):
        self.vertList = {}      # 字典，存储图中的所有顶点
        self.numVertices = 0    # 顶点的数量

    def addVertex(self, key):
        self.numVertices = self.numVertices + 1
        newVertex = Vertex(key)  # 创建一个新的顶点对象
        self.vertList[key] = newVertex  # 将新顶点添加到图中
        return newVertex
    
    def getVertex(self, n):
        if n in self.vertList:
            return self.vertList[n]
        else:
            return None
        
    def __contains__(self, n):      # 经典魔法方法，支持in操作符
        return n in self.vertList
    
    def addEdge(self, fromV, toV, weight=0):
        if fromV not in self.vertList:
            newFromV = self.addVertex(fromV)  # 如果起点不存在，先创建它
        if toV not in self.vertList:
            newToV = self.addVertex(toV)      # 如果终点不存在，先创建它
        self.vertList[fromV].addNeighbor(self.vertList[toV], weight)
        # 如果想实现无向图，下面这行代码也要加上使其双向
        # self.vertList[toV].addNeighbor(self.vertList[fromV], weight)

    def getVertices(self):
        return self.vertList.keys()  # 返回图中所有顶点的编号/名字
    
    def __iter__(self):
        return iter(self.vertList.values())  # 迭代器，返回图中所有顶点对象，可以写`for v in g:`来遍历图中的所有顶点

### 实例一：Word Ladder Graph
![](/img/python/word-ladder-graph.png)
所谓的单词接龙，说白了就是小学生问题，适合发呆的时候在课桌或者草稿本上玩的游戏。
给定一个起点单词FOOL，要求每次只改变一个字母的前提下（每次变化后得到的仍然必须是一个合法单词），到达重点SAGE。
在一堆单词之间寻找一条 **最短变换路径**，最好的方法将就是我们已经全局建好这样的 **图**。

- 每个单词都是图中的一个节点
- 如果两个单词只差一个字母，那就在它们之间连一条边
- 默认每一步变化的“代价”都一样，所以是无权图
#### 建图
最直接的建图方法就是 **两两比较**。
- 先把所有单词都变成顶点
- 然后把每个单词和其他所有单词逐一比较
- 如果只差一个字母，就连边

这个方法很容易想象，就是遍历邻接矩阵，复杂度O(n^2)。
但是如果有 5110 个四字母单词，那么两两比较会超过 2600 万次。这就是典型的平方级暴力！！

更聪明的方法是桶（bucket）思想 —— 不要直接比较对象和对象，而是先给对象构造“特征分组”。
![](/img/python/word-buckets.png)
##### 什么是桶
对于一个四字母单词，比如 pole，我们每次挖掉一个位置，用下划线 _ 代替，就能生成若干个“模式”：
- _ole
- p_le
- po_e
- pol_

这些模式就可以看成“桶的标签”。如果两个单词只差一个字母，那么它们一定会落入某个相同的桶里。
pope、pops它们都能匹配 pop_
一旦同桶，就说明它们之间只差一个字母，应该连边。
##### 桶法建图
第一步：给每个单词建顶点
第二步：为每个单词生成多个“缺一位”的模式
第三步：用字典把模式映射到单词列表
```
{
    "_ool": ["fool", "pool", "cool"],
    "fo_l": ["fool", "foil"],
    "pol_": ["poll", "pole"]
}
```
  - key：某种“只缺一个字母”的模式
  - value：所有匹配这个模式的单词

In [ ]:
def buildGraph(wordFile):
    d = {}      # 辅助结构桶字典
    g = Graph()
    wfile = open(wordFile, 'r')

    for line in wfile:
        word = line[:-1] # 切片去掉换行符
        for i in range(len(word)):
            bucket = word[:i] + '_' + word[i+1:]    # 生成桶的名字，例如对于"FOOL"，当i=0时，桶名为"_OOL"，当i=1时，桶名为"F_OL"，长度为 m 的单词，就会生成 m 个桶
            if bucket in d:
                d[bucket].append(word)              # 若桶名已存在：将当前单词追加到该桶的单词列表
            else:
                d[bucket] = [word]                  # 若桶名不存在：以桶名为键，创建新列表存储当前单词
    # 上面这一步只是更新字典d，不涉及单词

    for bucket in d.keys():
        for word1 in d[bucket]:
            for word2 in d[bucket]:
                if word1 != word2:
                    g.addEdge(word1, word2)  # 在图中添加边，表示word1和word2只有一个字母不同

    return g

#### 最短路径 BFS
这是一个无权图最短路径问题，所以应该优先想到 **BFS（广度优先搜索）**
DFS 的特点是：
- 一条路走到黑
- 尽量往深处钻
所以 DFS 能找到一条路径，但通常不能保证最短。
与此同时，BFS 的关键性质是它会先找到距离起点为 k 的所有节点，再去找距离为 k+1 的节点，像一层一层往外扩散：
- 第 0 层：起点
- 第 1 层：与起点一步可达的单词
- 第 2 层：两步可达的单词
- 第 3 层：三步可达的单词

##### 额外维护的三个属性
顶点类除了基本图结构，还增加了三个变量：
- distance
  - 从起点到当前顶点的最短距离（边数）
- predecessor（前驱）
  - 当前顶点是从哪个顶点第一次被发现的
  - 方便最后回溯整条最短路径
- color
  - white：表示还没有被发现
  - gray：表示已经发现，但还没完全处理完
  - black：表示这个点的所有邻接点都已经检查完了
  - 防止：重复访问；重复入队；在有环图中反复绕圈
![](/img/python/bfs-1.png)
![](/img/python/bfs-2.png)
![](/img/python/bfs-3.png)
![](/img/python/bfs-4.png)

上面的流程构造了一棵“广度优先树”
因为 BFS 是按层推进的。
- 当某个点第一次入队时，它一定是通过“当前最短层数”到达的
- 以后即使还有其他路径能到它，也不会更短
所以 BFS 中一个点第一次被发现时，它的最短距离就已经确定了。


In [ ]:
def bfs(g, start):
    # 对起点 start：
    start.distance = 0
    start.pred = None
    query.enqueue(start)

    while queue is not empty:
        currentVert = queue.dequeue()

        for nbr in currentVert.getConnections():
            if nbr.color == 'white':  # 如果这个邻居还没有被访问过
                nbr.color = 'gray'  # 标记为正在访问
                nbr.distance = currentVert.distance + 1  # 更新距离
                nbr.pred = currentVert  # 设置前驱节点
                queue.enqueue(nbr)  # 将邻居加入队列

        currentVert.color = 'black'  # 标记为访问完成

def traverse(y):
    x = y
    while x.pred is not None:
        print(x.id)
        x = x.pred
    print(x.id)  # 打印起点的id

#### 复杂度分析
- 入队：
BFS 中，队列里的每个顶点最多只会出队一次。
一个顶点必须先是白色（未访问），才会被发现并加入队列。
因此，while 循环最多执行 |V| 次。
- 出队：
而每次从队列中取出一个顶点 u，都会遍历 u 的邻接点。
从整体来看，所有顶点的邻接边总共只会被检查一遍，因此内层循环总成本与边数成正比。

因此把两部分合起来，时间复杂度为 O(V + E)
- 顶点数越多，搜索成本越大
- 边数越多，遍历邻接关系的成本越大

BFS 搜完以后，还要从目标节点沿着 predecessor 一直回溯到起点。
**最坏情况** 下，如果整张图退化成一条长链，那么需要经过所有顶点。
路径回溯复杂度写作 O(V)

> 当然上述没有包括建图成本

### 实例二：Knight’s Tour Problem
骑士周游是国际象棋中的经典问题：
- 棋盘上只有一个骑士
- 要找到一条走法序列
- 使骑士恰好访问棋盘上的每一个格子一次
- 并且不重复访问任何格子
如果存在这样一条路径，就称为一个 tour（巡游）。
#### 建图
把棋盘上的每一个格子都看作一个顶点；如果骑士能够从格子 A 一步合法跳到格子 B， 那么就在 A 和 B 之间建立一条边。
骑士周游问题就转化为 **在图中寻找一条路径，使每个顶点恰好访问一次**
目标路径长度应为 rows x columns - 1。
![](/img/python/knight-legal-moves.png)
> BTW 我不了解国际象棋，骑士好像是个马啊，横走 1，竖走 2 或横走 2，竖走 1

![](/img/python/knight-legal-moves-all.png)

In [ ]:
def posToNodeId(row, col, bdSize):
    return (row * bdSize) + col  # 将棋盘上的位置(row, col)转换为图中的节点ID
    # 例如在 8 × 8 棋盘中：
    # (0,0) -> 0
    # (0,1) -> 1
    # (1,0) -> 8

def knightGraph(bdSize):
    ktGraph = Graph()
    for row in range(bdSize):
        for col in range(bdSize):
            nodeId = posToNodeId(row, col, bdSize)  # 将棋盘上的位置转换为图中的节点ID
            newPositions = genLegalMoves(row, col, bdSize)  # 生成骑士在当前位置可以合法移动到的所有位置
            # 骑士共有 8 种理论移动方式：
            # (-1,-2), (-1,2), (-2,-1), (-2,1),
            # ( 1,-2), ( 1,2), ( 2,-1), ( 2,1)

            for e in newPositions:
                mod = posToNodeId(e[0], e[1], bdSize)  # 将合法移动的位置转换为图中的节点ID
                ktGraph.addEdge(nodeId, mod)  # 在图中添加边，表示骑士可以从nodeId位置移动到mod位置
    return ktGraph


def genLegalMoves(x, y, bdSize):
    newMoves = []
    moveOffsets = [(-1,-2), (-1,2), (-2,-1), (-2,1), (1,-2), (1,2), (2,-1), (2,1)]
    for i in moveOffsets:
        newX = x + i[0]
        newY = y + i[1]
        if legalCoord(newX, bdSize) and legalCoord(newY, bdSize):
            newMoves.append((newX, newY))  # 如果新位置合法，将其添加到合法移动列表中
    return newMoves

def legalCoord(x, bdSize):
    if x >= 0 and x < bdSize:
        return True
    else:        return False



#### 路径搜索 DFS
骑士周游要求的是找一条经过所有顶点且不重复的路径。
这不是普通“最短路径”问题，而是一个约束很强的路径搜索问题
因此更自然的思路是：
- 沿着一条路径不断深入尝试
- 如果走不通，再退回来换一条路
这正是 DFS（深度优先搜索） 的典型风格。
> 这里我们依旧保留BFS的白灰（没有黑）
> - white = 未被占用（可以走）
> - gray = 正在当前路径中（被占用，不能重复走）
> - black 不能有啊，因为每一个节点都必须访问到

In [ ]:
def knightTour(n, path, u, limit):
    # n：当前深度（搜索树中的层数）
    # path：到目前为止已经走过的顶点序列
    # u：当前正在探索的顶点
    # limit：目标深度上限，对于 8 × 8 棋盘： 共有 64 个格子 目标深度是 63

    u.setColor('gray')  # 把当前顶点染成灰色，表示正在访问
    path.append(u)      # 这个点已经在当前路径上了

    if n < limit:       # 递归辅助入口，第一遍执行可以忽视这个外壳
        nbrList = list(u.getConnections())  # 获取当前顶点的所有邻居
        i = 0           # 从第一个邻居开始深搜
        done = False
        while i < len(nbrList) and not done:
            if nbrList[i].getColor() == 'white':  # 递归的唯一入口！只有white（未被占用）的节点，才有资格进入递归
                done = knightTour(n+1, path, nbrList[i], limit) # 递归调用，继续探索下一个顶点
                # 这里n+1表示深度走了一步
                # 邻居节点nbrList[i]作为新的当前节点u
                # 进入递归后，第一行代码立刻标为 gray（占用）
            i = i + 1
        
        if not done:
            path.pop()  # 回退，移除当前顶点
            u.setColor('white')  # 把当前顶点染回白色，表示未访问，允许上层递归重新选择
    else:
        done = True  # 已经达到目标深度，成功找到一条路径
    return done

# 1. 访问当前顶点
# 2. 标记已访问，加入路径
# 3. 若路径长度达到要求，返回成功
# 4. 否则继续递归探索一个白色邻居
# 5. 若所有邻居都失败，则回溯

> 代码怎么保证上一层递归不会重复滥用这个节点？
> 这里有个i表示搜索树的编号

这里我们还没有比较BFS的队列和DFS的栈。
BFS 使用 Queue（队列）原因是先进先出：
- 先发现的顶点先处理
- 距离近的层先被完全处理
- 然后才会处理更远的层
因此 BFS 才能保证“按层搜索”。
如果不用队列会怎样？改成栈，就会优先向深处走，变成 DFS 风格，就不能保证先找到最短路径。

DFS 这里没有显式写栈，但实际上递归本身就在使用系统调用栈。因此：
- 向下递归 = 入栈
- 返回上一层 = 出栈
- 回溯 = 借助递归栈恢复现场
这就是“DFS 天然适合回溯搜索”的原因。

这类问题本质上是 **枚举路径 + 剪枝 + 回溯**

#### 复杂度分析
##### 搜索树
![](/img/python/knight-tour-search-tree.png)
骑士周游的难点不在建图，而在搜索。每走一步，接下来都可能有多个合法下一步，因此搜索会形成一棵巨大的搜索树。
搜索树的特点：
- 根节点：起点
- 每一层：一次新的落子选择
- 每个节点的孩子数：当前位置可走的下一步数量
##### 分支因子
骑士在不同位置可走步数不同：
- 角落：2
- 靠边：3 或 4
- 中间：最多 8
因此每个节点的分支数不是固定的，通常用一个平均分支因子（branching factor） k 来近似估计。
![](/img/python/knight-tour-branching-factor.png)

如果棋盘有 N 个格子，则搜索树深度大约为 N。
每层平均有 k 个分支，因此算法规模是指数级的 O(k^N)
因为问题通常有多个解，搜索一旦找到某个合法 tour 就可以提前停止。但即便如此，能少搜一些，只是常数倍上的改进，不能改变其指数级本质。

| 棋盘规格 | 搜索深度 | 平均分支因子(k) | 搜索树节点规模 |
| -------- | -------- | --------------- | -------------- |
| 5 × 5    | 25 层    | 3.8             | ≈ 3.12×10¹⁴    |
| 6 × 6    | 35 层    | 4.4             | ≈ 1.5×10²³     |
| 8 × 8    | 63 层    | 5.25            | ≈ 1.3×10⁴⁶     |

##### 启发式策略：Warnsdorff’s Rule
优先走“可选后续最少”的那个顶点。

In [ ]:
def orderByAvail(n):
    resList = []
    # --------------- 第一层循环：遍历【当前节点n】的所有邻居 v ---------------
    for v in n.getConnections():
        if v.getColor() == 'white':
            c = 0
            # ------------ 第二层循环：遍历【邻居v】的所有邻居 w ------------
            for w in v.getConnections():
                if w.getColor() == 'white':
                    c = c + 1       # 这个邻居还能走到多少个白色顶点为c
            resList.append((c, v))  # 将邻居和它的可用邻居数量作为一个元组(可用步数, 顶点)添加到结果列表中
    resList.sort(key=lambda x: x[0])  # 根据可用邻居数量进行排序，优先访问可用邻居较少的节点
    return [v for (c, v) in resList]  # 返回排序后的邻居列表，去掉数量信息

# 这里确实c只有2层

如果过早进入棋盘中间那些“连接很多”的位置，就容易：
- 把中间通道提前用掉
- 最后被困在边角
- 导致剩余某些格子再也到不了
边缘和角落位置合法走法少，更难覆盖。因此应该优先访问这些“低度数顶点”，从而：
- 提前解决难处理区域
- 把中间高连接度格子留作后续跳板
- 大幅减少走到死路的概率

这类方法叫 **启发式（heuristic）**，即利用问题本身的结构特征，帮助算法更快地做出更靠谱的选择。
它不保证从数学上减少复杂度阶数，但在实际运行中往往能极大提升效率。

这种“优先访问后续可选步数最少的点”的规则叫 `Warnsdorff’s algorithm`，先走最受限制的位置。
这是骑士周游问题中最经典的启发式策略之一。使用它后，8 × 8 棋盘的求解速度会显著提升。

### 通用深度优先搜索
骑士周游里的 DFS 是一种特殊情况，它希望得到一条尽可能深、且不分叉的路径。
而一般的 DFS 更通用，它的目标是：
- 从某个顶点出发，尽可能向深处搜索
- 能连通多少顶点就连通多少顶点
- 必要时允许分支
- 最终构造出一棵或多棵深度优先树

如果整张图不是连通的，那么从一个起点出发不一定能访问所有顶点。因此，一般 DFS 不只会生成一棵树，而可能生成多棵深度优先树
这些树的集合叫 **深度优先森林（depth first forest）**
#### 两个额外的顶点信息
除了前面学过的color、predecessor，一般 DFS 还会新增两个时间戳变量：
- discovery time：发现时间
  - 一个顶点第一次被访问到时，算法进行到第几步
- finish time：完成时间
    - 一个顶点的所有邻接顶点都探索完成、自己被染成黑色时，算法进行到第几步

这两个时间不是“为了记录而记录”，而是后面很多算法的基础：
- 拓扑排序靠 finish time
- 强连通分量算法也靠 finish time
- DFS 树中父子节点之间还会表现出特殊的时间嵌套关系



In [ ]:
class DFSGraph(Graph):
    def __init__(self):
        super().__init__()
        self.time = 0  # 用于记录访问时间的计数器

    def dfs(self):
        for aVertex in self:
            aVertex.setColor('white')  # 初始化所有顶点为未访问状态
            aVertex.setPred(-1)     # 初始化所有顶点的前驱节点为None

        for aVertex in self:
            if aVertex.getColor() == 'white':  # 如果顶点未被访问过
                self.dfsVisit(aVertex)  # 从该顶点开始进行DFS访问

    def dfsVisit(self, startVertex):
        startVertex.setColot('gray')
        self.time = self.time + 1
        startVertex.setDiscovery(self.time)  # 记录发现时间
        for nextVertex in startVertex.getConnections():
            if nextVertex.getColor() == 'white':
                nextVertex.setPred(startVertex)  # 设置前驱节点
                self.dfsVisit(nextVertex)  # 递归访问下一个顶点
        startVertex.setColor('black')
        self.time = self.time + 1
        startVertex.setFinish(self.time)  # 记录完成时间

#### 基本流程
对于某个顶点 u：
1. 发现 u
2. 记录 u.discovery
3. 尝试递归访问它的白色邻接点染成灰色，时间相对增加
4. 所有邻接点都处理完后
5. 当前节点染成黑色，记录 u.finish


```
      A(灰,1)
     ╱    ╲
  B(白)   D(白)
   │        │
  C(白)   E(白)
            │
          F(白)
            │
          C(白)

【状态】A标记灰色，发现时间=1
```
```
      A(灰,1)
     ────
    B(灰,2)
   ╱    ╲
C(白)   D(白)
         │
       E(白)
         │
       F(白)
         │
       C(白)

【状态】B标记灰色，发现时间=2
```
```
      A(灰,1)
     ────
    B(灰,2)
   ────
C(灰,3)

D(白)──E(白)──F(白)──C(灰,3)

【状态】C标记灰色，发现时间=3
```
```
      A(灰,1)
     ────
    B(灰,2)
   ────
C(黑,3,4)  ← 完成时间=4

D(白)──E(白)──F(白)

【状态】C无邻居，完成访问
```
```
      A(灰,1)
     ────
    B(灰,2)
   ────  ╲
C(黑,3,4) D(灰,5)
            │
          E(白)
            │
          F(白)

【状态】D标记灰色，发现时间=5
```
```
      A(灰,1)
     ────
    B(灰,2)
   ────  ╲
C(黑,3,4) D(灰,5)
            ────
              E(灰,6)
             ╱    ╲
   B(灰,2)←虚线   F(白)

【状态】E标记灰色，发现时间=6；B已灰色，虚线跳过
```
```
      A(灰,1)
     ────
    B(灰,2)
   ────  ╲
C(黑,3,4) D(灰,5)
            ────
              E(灰,6)
                    ────
                      F(灰,7)
                       │
             C(黑,3,4)←虚线

【状态】F标记灰色，发现时间=7；C已黑色，虚线跳过
```
```
      A(灰,1)
     ────
    B(灰,2)
   ────  ╲
C(黑,3,4) D(灰,5)
            ────
              E(灰,6)
                    ────
                      F(黑,7,8)  ← 完成时间=8

【状态】F无新邻居，完成访问
```
```
      A(灰,1)
     ────
    B(灰,2)
   ────  ╲
C(黑,3,4) D(灰,5)
            ────
              E(黑,6,9)  ← 完成时间=9
                    ────
                      F(黑,7,8)

【状态】E所有邻居处理完毕，完成访问
```
```
      A(灰,1)
     ────
    B(灰,2)
   ────  ╲
C(黑,3,4) D(黑,5,10)  ← 完成时间=10
            ────
              E(黑,6,9)
                    ────
                      F(黑,7,8)

【状态】D所有邻居处理完毕，完成访问
```
```
      A(灰,1)
     ────
    B(黑,2,11)  ← 完成时间=11
   ────  ╲
C(黑,3,4) D(黑,5,10)
            ────
              E(黑,6,9)
                    ────
                      F(黑,7,8)

【状态】B所有邻居处理完毕，完成访问
```
```
      A(黑,1,12)  ← 完成时间=12
     ────
    B(黑,2,11)
   ────  ╲
C(黑,3,4) D(黑,5,10)
            ────
              E(黑,6,9)
                    ────
                      F(黑,7,8)

【状态】DFS全图遍历完成
```
```
        A
       /
      B
     / \
    C   D
        |
        E
        |
        F

【树边】A→B, B→C, B→D, D→E, E→F
【回边】E→B, F→C (虚线，非树边)
```
#### 括号性质（Parenthesis Property）
DFS 中，顶点的发现时间和完成时间满足一个很重要的性质：
**一个节点的所有子节点，其发现时间一定晚于父节点，完成时间一定早于父节点**

也就是说：
- 父节点先被发现
- 子节点在父节点探索过程中被发现
- 子节点先完成
- 父节点最后完成

如果把每个顶点看成一个 **时间区间**，父节点的时间区间会“包住”所有子节点的区间。例如：
- 父节点：[2, 11]
- 子节点：[3, 4]
- 子节点：[5, 10]
这就是所谓的“括号性质”。它说明 DFS 树中的父子关系，在时间戳上会自然形成嵌套结构。
这个性质后面在一些图论算法分析中非常重要。


#### 再次对比深搜广搜
| 类别/对比维度 | BFS | DFS |
| :--- | :--- | :--- |
| **共同点** | 均使用颜色标记节点状态（白/灰/黑） | 同左 |
| （两者均具备） | 均记录前驱节点（`prev`） | 同左 |
|  | 均构造搜索树 / 搜索森林 | 同左 |
|  | 均避免节点重复访问 | 同左 |
| 核心数据结构 | 队列（Queue） | 递归调用栈（隐式栈） |
| 遍历方式 | 按层扩散、先近后远 | 一路到底、深度优先 |
| 核心特性 | 层序遍历、天然最短路径 | 回溯、深度优先探索 |
| 典型适用场景 | 无权图最短路径、层序遍历 | 连通性分析、拓扑排序、强连通分量、回溯搜索 |

#### 复杂度分析
1. dfs() 中两次 for aVertex in self 都是对所有顶点扫描一次：O(V)
2. dfsvisit() 中遍历邻接边的成本
   - 每个顶点最多只会被真正递归访问一次
   - 每条边最多只会被检查一次（或常数次）
   - 因此边的总成本为 O(E)
因此一般 DFS 的标准复杂度为 O(V+E)，这与邻接表实现下的 BFS 一样，也是图搜索的经典复杂度。
### 拓扑排序（Topological Sorting）
拓扑排序只适用于 **有向无环图（DAG, Directed Acyclic Graph）**。如果图中有环，就不存在合法的拓扑序。
定义为给定一个 DAG，拓扑排序要产生一个线性序列，使得如果图中存在边 (v,w)，那么在序列中 v 一定出现在 w 前面。
本质上是在解决有依赖关系的任务，应该按什么顺序执行的问题。
- 做菜步骤
- 软件项目依赖
- 编译顺序
- 课程先修关系
- 数据库查询优化
- 矩阵链乘法中的依赖关系

![煎饼图上深度优先搜索森林的结果](/img/python/pancake-graph-dfs.png)
如上，因为 DFS 会给每个节点一个 finish time。
而在 DAG 中，一个节点的后继节点通常会比它更早完成。
因此 **按 finish time 从大到小排列顶点，就能得到一个合法的拓扑序**。
#### 伪代码
```
// 全局/类变量
time = 0                // 全局时间戳
result_list = []        // 存储拓扑排序结果

// 顶点状态常量
WHITE = "未访问"
GRAY = "访问中"
BLACK = "访问完成"

// ------------------------------
// 1. DFS 遍历：计算顶点完成时间
// ------------------------------
FUNCTION dfs(graph):
    // 初始化所有顶点：未访问、无前驱
    FOR each vertex IN graph.vertices:
        vertex.color = WHITE
        vertex.pred = NULL
    
    // 遍历所有顶点，启动DFS
    FOR each vertex IN graph.vertices:
        IF vertex.color == WHITE:
            dfs_visit(vertex)

// DFS 递归辅助函数
FUNCTION dfs_visit(vertex):
    vertex.color = GRAY          // 标记为访问中
    time = time + 1
    vertex.discovery = time      // 记录发现时间

    // 遍历当前顶点的所有邻接顶点
    FOR each neighbor IN vertex.get_neighbors():
        IF neighbor.color == WHITE:
            neighbor.pred = vertex
            dfs_visit(neighbor)
    
    // 所有子节点遍历完成，标记为已完成
    vertex.color = BLACK
    time = time + 1
    vertex.finish = time          // 记录完成时间

// ------------------------------
// 2. 拓扑排序主函数（核心）
// ------------------------------
FUNCTION topological_sort(graph):
    // 步骤1：执行DFS，计算所有顶点的完成时间
    dfs(graph)
    
    // 步骤2：按【完成时间降序】排序所有顶点
    sorted_vertices = SORT(graph.vertices) BY vertex.finish IN DESCENDING ORDER
    
    // 步骤3：提取排序后的顶点，返回拓扑序
    result_list = [vertex.id FOR vertex IN sorted_vertices]
    
    RETURN result_list
```
![拓扑排序 = 对 DAG 做 DFS + 按完成时间逆序输出](/img/python/topological-sort-result.png)

### 强连通分量（Strongly Connected Components, SCC）
对于大型有向图（例如网页链接图），我们经常希望找出哪些顶点彼此之间是“互相可达”的。
这类紧密互联的顶点集合，就是强连通分量。
设图 G = (V, E)，若顶点集合 C⊆V 满足：
- 对任意 v, w∈C 都存在：
    - 从 v 到 w 的路径
    - 从 w 到 v 的路径
那么 C 就是一个强连通分量。
![含三个强连通分量的有向图](/img/python/strongly-connected-components.png)
> “最大子集”这个条件很重要，不是随便找一个互相可达的小集合就行，再加别的点就不满足互相可达了

强连通分量可以帮助我们：
- 发现图中的“团块结构”
- 压缩复杂图
- 分析网页聚类
- 分析调用关系
- 处理依赖系统中的循环结构

当我们找到所有 SCC 之后，可以把每个强连通分量压缩成一个“大顶点”，这样得到的新图通常会更简单。
这种过程通常叫 **缩点**，或 **得到分量图 / reduced graph**。
压缩后的图会更清楚地展示不同强连通块之间的依赖关系。
![](/img/python/reduced-graph.png)

> 这里定义一下图的转置（Transpose）
> 也就是把有向图的方向全部反过来
> 显而易见一个图和他的转置图，强联通分量是相同的
> 原来互相可达的一组点，仍然互相可达

#### Kosaraju 算法
首先在原图 G 上做 DFS，计算每个顶点的 finish time；
接着构造转置图，把所有边反向，再做一次 DFS（主循环要按照原图 finish time 从大到小选择起点）
第二次 DFS 得到的森林中每一棵树就是一个强连通分量，把每棵树中的顶点输出出来即可。
```
// 全局变量定义
time = 0                    // 时间戳，记录发现/完成时间
finish_order = []           // 存储原图DFS的顶点【完成时间升序】
scc_list = []               // 存储最终的所有强连通分量
WHITE = 未访问
GRAY  = 访问中
BLACK = 访问完成

// ======================================
// 基础DFS函数：用于原图，计算完成时间
// ======================================
FUNCTION dfs(vertex):
    vertex.color = GRAY
    time = time + 1
    vertex.discovery = time
    
    // 遍历所有邻接节点
    FOR each neighbor IN vertex.adjacent:
        IF neighbor.color == WHITE:
            dfs(neighbor)
    
    vertex.color = BLACK
    time = time + 1
    vertex.finish = time
    finish_order.APPEND(vertex)  // 记录完成顺序

// ======================================
// 转置图函数：反转图G所有边，得到 G^T
// ======================================
FUNCTION transpose_graph(G):
    GT = 空图
    // 复制所有顶点
    FOR each vertex IN G.vertices:
        GT.add_vertex(vertex.id)
    // 反转所有有向边
    FOR each vertex IN G.vertices:
        FOR each neighbor IN vertex.adjacent:
            GT.add_edge(neighbor.id, vertex.id)
    RETURN GT

// ======================================
// DFS遍历函数：用于转置图，生成SCC
// ======================================
FUNCTION dfs_scc(vertex, current_component):
    vertex.color = GRAY
    current_component.APPEND(vertex.id)  // 加入当前强连通分量
    
    FOR each neighbor IN vertex.adjacent:
        IF neighbor.color == WHITE:
            dfs_scc(neighbor, current_component)
    
    vertex.color = BLACK

// ======================================
// 强连通分量主算法
// ======================================
FUNCTION strongly_connected_components(G):
    // 步骤1：对原图 G 执行DFS，计算完成时间
    初始化所有顶点为 WHITE
    time = 0
    finish_order = []
    FOR each vertex IN G.vertices:
        IF vertex.color == WHITE:
            dfs(vertex)

    // 步骤2：计算转置图 G^T
    GT = transpose_graph(G)

    // 步骤3：对 GT 执行DFS，按【完成时间降序】遍历顶点
    初始化 GT 中所有顶点为 WHITE
    scc_list = []
    // 反转完成顺序 → 降序
    reverse_order = REVERSE(finish_order)
    
    FOR each vertex IN reverse_order:
        IF GT.get_vertex(vertex.id).color == WHITE:
            current_component = []
            // 递归遍历，生成一棵DFS树 = 一个SCC
            dfs_scc(GT.get_vertex(vertex.id), current_component)
            scc_list.APPEND(current_component)

    // 步骤4：输出所有强连通分量
    RETURN scc_list
```

### 最短路径问题
现实中的很多网络传输问题，本质上都可以抽象成图问题。
例如：
- 网页请求从本地电脑发到服务器
- 邮件从一台主机发送到另一台主机
- 数据包在多个路由器之间转发

这些过程背后都涉及若干个中间节点、若干条连接路径、每条连接有不同代价。
因此可以将其抽象成一个 **带权图**：
- 顶点（Vertex）：路由器、主机、网络节点
- 边（Edge）：两个节点之间的连接
- 权重（Weight）：走这条连接的代价

最短路径问题的定义就是：给定一个带权图和起点 s，希望找到 **从 s 到其他顶点的总代价最小路径**。

> 如果图中所有边权重都相同，那么此时问题就退化成无权图最短路径，方法则选用 **BFS**

#### Dijkstra 算法
Dijkstra 算法用于解决 **单源最短路径问题（Single Source Shortest Path）**：
即给定一个起点，求它到图中所有其他顶点的最短路径。
> 一旦存在负权边，Dijkstra 就不能保证结果正确，所以首先要判断图里是否可能出现 **负权**
基本思想是每次都从“当前已知距离最小”的那个顶点出发，继续向外更新其他顶点的最短距离
> 这是一种典型的 **贪心（greedy）思想**



In [ ]:
def dijkstra(aGraph, start):
    # 1. 初始化：起点距离设为0，其余顶点默认为无穷大
    start.setDistance(0)

    # 2. 最小堆，距离越小优先级越高
    pq = PriorityQueue()  # 创建一个优先队列来存储顶点，按照距离排序
    pq.buildHeap([(v.getDistance(), v) for v in aGraph])  # 将图中的所有顶点添加到优先队列中
    # (距离, 顶点) 的元组

    # 3. 核心循环：取出距离最小的顶点
    while not pq.isEmpty():
        # 4. 遍历当前顶点的所有邻居
        currentVert = pq.delMin() # 出堆顶（当前最短路径顶点），起点默认在堆顶

        for nextVert in currentVert.getConnections():
            # 计算：起点 -> current -> next 的新路径长度
            newDist = currentVert.getDistance() + currentVert.getWeight(nextVert)

            if newDist < nextVert.getDistance():  # 如果新距离更短
                nextVert.setDistance(newDist)  # 更新邻居顶点的距离
                nextVert.setPred(currentVert)  # 更新邻居顶点的前驱节点
                pq.decreaseKey(nextVert, newDist)  # 更新优先队列中的距离信息

示例图（无负权边，Dijkstra 标准场景）
- 顶点：A(起点)、B、C；
- 边（有向 + 权重）：
    - A → B 权重 = 2
    - A → C 权重 = 5
    - B → C 权重 = 1
目标：求起点 A 到所有点的最短路径

**第一次循环**：
最小堆：`[(0,A), (∞,B), (∞,C)]`
currentVert = pq.delMin() → 取出 A
遍历邻居：
    处理邻居 B：
        newDist = 0 + 2 = 2
        判断：2 < ∞ → 成立
        B.setDistance(2)
        B.setPred(A)
        堆中更新 B 的距离
    处理邻居 C
        newDist = 0 + 5 = 5
        判断：5 < ∞ → 成立
        C.setDistance(5)
        C.setPred(A)
堆剩余：`[(2,B), (5,C)]`

**第二次循环**：
currentVert = pq.delMin() → 取出 B（距离 2）
遍历邻居：
    处理邻居 C
        newDist = 2 + 1 = 3
        判断：3 < 5 → 成立
        C.setDistance(3)
        C.setPred(B)
堆剩余：`[(3,C)]`

**第三次循环**：
currentVert = pq.delMin() → 取出 C
C 无邻居，循环结束

> 在互联网上使用迪杰斯特拉算法的一个问题是，必须拥有完整的图表示才能运行该算法。这意味着每个路由器都需要掌握互联网中所有路由器的完整映射。但实际情况并非如此，该算法的其他变体允许每个路由器逐步探索图结构。可以了解一种名为“距离矢量”路由算法的相关算法。

#### 复杂度分析
这里默认采用：
- 邻接表
- 二叉堆实现的优先队列

1. 建堆：把所有顶点加入优先队列 O(V)
2. 主循环的 delMin：
   - 每个顶点最多出队一次，共 V 次；
   - 每次取最小值 delMin，堆要重新调整，耗时 O(logV)
   - 总计 O(VlogV)
3. 遍历所有边 + decreaseKey：
   - 每条边最多引发一次 decreaseKey（更新距离），共E次
   - 每次更新堆 decreaseKey 耗时 O(logV)
   - 总计 O(ElogV)
4. 因此 Dijkstra 总复杂度为：O((V + E)logV)

### 最小生成树问题
有时我们的目标不是“从一个点到另一个点的最短路径”，而是用尽量小的总代价，把所有点连起来。
这就是 **最小生成树（Minimum Spanning Tree, MST）** 问题。

例如广播消息时，如果发送者对每个听众都单独发一份消息，会产生大量重复流量。如果采用“无控制泛洪”，又会造成更多冗余转发。
因此更合理的目标是找到一棵代价最小、又能覆盖所有顶点的树。
这样：
- 每条消息只需沿树传播一次
- 不会形成环
- 不会重复转发太多份

对于图 G = (V, E) ，其最小生成树 T 满足：
1. T 是 E 的一个子集 
2. T 连接了所有顶点 
3. T 没有环
4. T 中所有边的权重和最小

#### Prim 最小生成树算法
- 目标：用于求连通 **无向带权图** 的最小生成树
- 思路：从一个起点开始，不断把“当前树外、且与树相连的最便宜顶点” **贪心** 地接进来

Prim 每一步都要选一条 **安全边（safe edge）**，即一条连接“树中顶点”和“树外顶点”的边。这样做的好处是能把新顶点接进来，又不会形成环。

In [ ]:
def prim(G, start):
    pq = PriorityQueue()  # 最小堆，权重越小优先级越高
    for v in G:
        v.setDistance(float('inf'))
        v.setPred(None)

    start.setDistance(0)
    pq.buildHeap([(v.getDistance(), v) for v in G]) # 将图中的所有顶点添加到优先队列中

    while not pq.isEmpty():
        currentVert = pq.delMin() # 出堆顶（当前最小权重顶点）

        for nextVert in currentVert.getConnections():
            weight = currentVert.getWeight(nextVert)
            if weight < nextVert.getDistance():
                nextVert.setPred(currentVert)
                nextVert.setDistance(weight)
                pq.decreaseKey(nextVert, weight)


#### 对比 Dijkstra
| 对比维度 | Prim 算法 | Dijkstra 算法 |
| :--- | :--- | :--- |
| 数据结构 - 优先队列 | 使用优先队列 | 使用优先队列 |
| 辅助数组 - `distance` | 使用 `distance` 数组 | 使用 `distance` 数组 |
| 辅助数组 - `pred` | 使用 `pred` 前驱数组 | 使用 `pred` 前驱数组 |
| 算法流程 | 1. 从起点开始<br>2. 每次取优先队列中的最小顶点<br>3. 更新周围顶点 | 1. 从起点开始<br>2. 每次取优先队列中的最小顶点<br>3. 更新周围顶点 |
| **`distance[v]` 的含义** | 把顶点 `v` 接入当前生成树所需的**最小边权** | 从起点到 `v` 的**当前最短路径总长度** |
| **更新公式** | `new_cost = weight(current, next)` | `new_dist = current.dist + weight(current, next)` |
| **是否累加路径总代价** | **不累加**，只关心单边权重 | **累加**，计算全程总代价 |
| **核心目标** | 用最便宜的方式把点接进树里 | 找到从起点出发的最短路径 |

####